# Building a Tokenizer from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Byte-Level Encoding

The foundation. Convert any string into a sequence of bytes, map each byte to a printable character for display, and reverse the process.

In [ ]:
```python

def bytes_to_tokens(text):

    return list(text.encode("utf-8"))

def tokens_to_text(token_bytes):

    return bytes(token_bytes).decode("utf-8", errors="replace")

In [ ]:
```

Test on multilingual text to see the byte counts:

In [ ]:
```python

texts = [

    ("English", "hello"),

    ("Chinese", "你好"),

    ("Emoji", "🔥"),

    ("Mixed", "hello你好🔥"),

]

for label, text in texts:

    b = bytes_to_tokens(text)

    print(f"{label}: {len(text)} chars -> {len(b)} bytes -> {b}")

In [ ]:
```

"hello" is 5 bytes. "你好" is 6 bytes (3 per character). The fire emoji is 4 bytes. The byte-level tokenizer does not care what language it is. Bytes are bytes.

### Step 2: Pre-Tokenizer with Regex

Split text into chunks using the GPT-2 regex pattern. Each chunk gets tokenized independently by BPE.

In [ ]:
```python

import re

try:

    import regex

    GPT2_PATTERN = regex.compile(

        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

    )

except ImportError:

    GPT2_PATTERN = re.compile(

        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""

    )

def pre_tokenize(text):

    return [match.group() for match in GPT2_PATTERN.finditer(text)]

In [ ]:
```

The `regex` module supports Unicode property escapes (`\p{L}` for letters, `\p{N}` for numbers). The standard library `re` module does not, so we fall back to ASCII character classes. For production multilingual tokenizers, install `regex`.

Try it:

In [ ]:
```python

print(pre_tokenize("Hello, world! Don't stop."))

# [' Hello', ',', ' world', '!', " Don", "'t", ' stop', '.']

In [ ]:
```

The leading space stays attached to the word. Contractions split at the apostrophe. Punctuation becomes its own chunk. BPE will never merge tokens across these boundaries.

### Step 3: BPE on Byte Sequences

The core algorithm from Lesson 01, but now operating on pre-tokenized chunks independently.

In [ ]:
```python

from collections import Counter

def get_byte_pairs(chunks):

    pairs = Counter()

    for chunk in chunks:

        byte_seq = list(chunk.encode("utf-8"))

        for i in range(len(byte_seq) - 1):

            pairs[(byte_seq[i], byte_seq[i + 1])] += 1

    return pairs

def apply_merge(byte_seq, pair, new_id):

    merged = []

    i = 0

    while i < len(byte_seq):

        if i < len(byte_seq) - 1 and byte_seq[i] == pair[0] and byte_seq[i + 1] == pair[1]:

            merged.append(new_id)

            i += 2

        else:

            merged.append(byte_seq[i])

            i += 1

    return merged

In [ ]:
```

### Step 4: Special Token Handling

Special tokens need exact matching and fixed IDs. They bypass BPE entirely.

In [ ]:
```python

class SpecialTokenHandler:

    def __init__(self):

        self.special_tokens = {}

        self.pattern = None

    def add_token(self, token_str, token_id):

        self.special_tokens[token_str] = token_id

        escaped = [re.escape(t) for t in sorted(self.special_tokens.keys(), key=len, reverse=True)]

        self.pattern = re.compile("|".join(escaped))

    def split_with_specials(self, text):

        if not self.pattern:

            return [(text, False)]

        parts = []

        last_end = 0

        for match in self.pattern.finditer(text):

            if match.start() > last_end:

                parts.append((text[last_end:match.start()], False))

            parts.append((match.group(), True))

            last_end = match.end()

        if last_end < len(text):

            parts.append((text[last_end:], False))

        return parts

In [ ]:
```

### Step 5: Full Tokenizer Class

Chain everything together: normalize, split on special tokens, pre-tokenize, BPE merge, map to IDs.

In [ ]:
```python

import unicodedata

class ProductionTokenizer:

    def __init__(self):

        self.merges = {}

        self.vocab = {i: bytes([i]) for i in range(256)}

        self.special_handler = SpecialTokenHandler()

        self.next_id = 256

    def normalize(self, text):

        return unicodedata.normalize("NFKC", text)

    def train(self, text, num_merges):

        text = self.normalize(text)

        chunks = pre_tokenize(text)

        chunk_bytes = [list(chunk.encode("utf-8")) for chunk in chunks]

        for i in range(num_merges):

            pairs = Counter()

            for seq in chunk_bytes:

                for j in range(len(seq) - 1):

                    pairs[(seq[j], seq[j + 1])] += 1

            if not pairs:

                break

            best = max(pairs, key=pairs.get)

            new_id = self.next_id

            self.next_id += 1

            self.merges[best] = new_id

            self.vocab[new_id] = self.vocab[best[0]] + self.vocab[best[1]]

            chunk_bytes = [apply_merge(seq, best, new_id) for seq in chunk_bytes]

    def add_special_token(self, token_str):

        token_id = self.next_id

        self.next_id += 1

        self.special_handler.add_token(token_str, token_id)

        self.vocab[token_id] = token_str.encode("utf-8")

        return token_id

    def encode(self, text):

        text = self.normalize(text)

        parts = self.special_handler.split_with_specials(text)

        all_ids = []

        for part_text, is_special in parts:

            if is_special:

                all_ids.append(self.special_handler.special_tokens[part_text])

            else:

                for chunk in pre_tokenize(part_text):

                    byte_seq = list(chunk.encode("utf-8"))

                    for pair, new_id in self.merges.items():

                        byte_seq = apply_merge(byte_seq, pair, new_id)

                    all_ids.extend(byte_seq)

        return all_ids

    def decode(self, ids):

        byte_parts = []

        for token_id in ids:

            if token_id in self.vocab:

                byte_parts.append(self.vocab[token_id])

        return b"".join(byte_parts).decode("utf-8", errors="replace")

    def vocab_size(self):

        return len(self.vocab)

In [ ]:
```

### Step 6: Multilingual Test

The real test. Throw English, Chinese, emoji, and code at it.

In [ ]:
```python

corpus = (

    "The quick brown fox jumps over the lazy dog. "

    "The quick brown fox runs through the forest. "

    "Machine learning models process natural language. "

    "Deep learning transforms how we build software. "

    "def train(model, data): return model.fit(data) "

    "def predict(model, x): return model(x) "

)

tok = ProductionTokenizer()

tok.train(corpus, num_merges=50)

bos = tok.add_special_token("<|begin|>")

eos = tok.add_special_token("<|end|>")

test_texts = [

    "The quick brown fox.",

    "你好世界",

    "Hello 🌍 World",

    "def foo(x): return x + 1",

    f"<|begin|>Hello<|end|>",

]

for text in test_texts:

    ids = tok.encode(text)

    decoded = tok.decode(ids)

    print(f"Input:   {text}")

    print(f"Tokens:  {len(ids)} ids")

    print(f"Decoded: {decoded}")

    print()

In [ ]:
```

Chinese characters produce 3 bytes each. The emoji produces 4 bytes. None of these crash the tokenizer. None produce unknown tokens. That is the power of byte-level BPE.

## Exercises

In [ ]:
1. **Easy:** Add a `get_token_bytes(id)` method that shows the raw bytes for any token ID. Use it to inspect what your most common merged tokens actually represent.
2. **Medium:** Implement the Llama-style pre-tokenizer that splits on whitespace and digits but keeps leading spaces. Compare its vocabulary with the GPT-2 regex approach on the same corpus.
3. **Hard:** Add a chat template method that takes a list of `{"role": ..., "content": ...}` messages and produces the correct token sequence for the Llama 3 chat format. Test it against the HuggingFace implementation.